# More Datasets
This notebooks investigates further datasets

In [ ]:
# Setting everything to be deterministic
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import seaborn as sns
sns.set_theme(style="whitegrid")


import numpy as np
import random, torch, os, cv2

seed = 0

print(f"Using device: {torch.device("cuda" if torch.cuda.is_available() else "cpu")}")

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import matplotlib.pyplot as plt

# Import the headset_localization package (used for localizers)
from headset_localization import *
# Import the CompleteRobotScan from the data generation pipeline
from shared import CompleteRobotScan

In [ ]:
from headset_localization import *

# Creation of the Predictors
def predictors_for_dataset(intrinsic_mtx:np.ndarray)->list[GradableLocalizer]:
    points_light_glue = GradableLocalizer(
        creator= PnPLocalizer.get_creation_function(
            cam2_intrinsic_mtx=intrinsic_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(),
                crop_augmentations=[0.4]
            )
        ),
        name="PnP-LG"
    )

    points_loma = GradableLocalizer(
        creator= PnPLocalizer.get_creation_function(
            cam2_intrinsic_mtx=intrinsic_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndMatchLoMa('LoMaB128'),
                crop_augmentations=[0.2]
            )
        ),
        name="PnP-LoMa"
    )

    points_lines_light_glue = GradableLocalizer(
        creator=PnPLLocalizer.get_creation_function(
            cam2_intrinsic_mtx = intrinsic_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(),
                crop_augmentations=[0.4]
            ),
            cam1_line_generator=LineGenerator(visualize_cleanup=False),
            cam2_line_generator=LineGenerator(    
                line_cleanup_config = MultiPassLineMergingConfig(
                    passes=[
                    LineMerging2dConfig(max_angle_diff = 2, max_midpoint_dist = 3/850, max_endpoint_dist = 0.01, min_line_length = 10/850, use_pca=False),
                    LineMerging2dConfig(max_angle_diff = 3, max_midpoint_dist = 4/850, max_endpoint_dist = 0.02, min_line_length = 20/850, use_pca=False)
                    ]
                )
            ),
            line_matching_config = LineMatchingConfig(max_point_line_dist_px=20, better_factor=1.2),
            debug_visualize_matching = False
        ),
        name="PnP+L-LG"
    )

    yolo = YOLOv26Segmenter(
        "yoloe-26l-seg.pt", 
        prompts=[
            "cup", "fruit", "plate", "teddy", "ball", "tennisball", "pen", "lego", "brick", "duplo", 
            "pen", "sphere", "round object", "tool", "toy", "plastic object"
        ]
    )

    ellipsoids_light_glue = GradableLocalizer(
        creator=EllipsoidLocalizer.get_creation_function(
            cam2_intrinsic_mtx = intrinsic_mtx,
            matching_config=GaussianMatchingConfig(dummy_value=0.001),
            cam1_segmenter = yolo,
            cam2_segmenter=yolo,
            #ellipsoid_fitter = MVEEEllipsoidFitter(contamination=0.1, visualize=False),
            ellipsoid_fitter=LeastShellDistanceEllipsoidFitter(contamination=0.02, size_penalty=0.05, size_p_norm=4),            
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(),
                crop_augmentations=[0.4],
            ),
            ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2, max_color_dist=10),
            visualize_environment_generation = False,
            visualize_segmentation_masks = False,
            visualize_matching=False,
            visualize_pne_optimisation=False
        ),
        name="Ellipsoids"
    )

    return [points_light_glue, points_loma, points_lines_light_glue, ellipsoids_light_glue]

def grader_for_dataset(env:Scanned3dEnvironment,headset_recording:HeadsetRecording)->NPredictors1DatasetGrader:
    return NPredictors1DatasetGrader(
        gradable_pose_predictors=predictors_for_dataset(headset_recording.intrinsic_cam_mtx),
        headset_data = headset_recording,
        robot_env = env,
        compute_ray_intersection_error=True, use_tqdm_for_frames=True, use_tqdm_for_predictors=False
    )

def plot_from_grader(grader:NPredictors1DatasetGrader, vis:bool = False):
    if vis:
        grader.visualize_predictions_3d()
    grader.print_summary()
    _, ax = plt.subplots(1, 1, figsize = (12, 3))
    grader.plot_time_series_error(ax, TimeSeriesErrorType.ABS_TRANSLATIONAL)
    _, axes = plt.subplots(2, 3, figsize = (15, 8))
    grader.plot_signed_error_comparison(axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]], explain = True)

## Loading Round1

In [ ]:
round1_scan = CompleteRobotScan.from_folder("../example_datasets/round1/round1_scan")

round1_env = Scanned3dEnvironment.from_gathered_robot_data(
        round1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.1, use_depth_images_if_provided=True),
        est3d_xyz_icp_config = ICPAlignmentConfig()
)

round1_rec1 = HeadsetRecording.from_folder("../example_datasets/round1/round1_rec1")
round1_rec2 = HeadsetRecording.from_folder("../example_datasets/round1/round1_rec2")
round1_rec3 = HeadsetRecording.from_folder("../example_datasets/round1/round1_rec3")

## Loading Brick1

In [ ]:
brick1_scan = CompleteRobotScan.from_folder("../example_datasets/brick1/brick1_scan")

brick1_env = Scanned3dEnvironment.from_gathered_robot_data(
        brick1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.1, use_depth_images_if_provided=True),
        est3d_xyz_icp_config = ICPAlignmentConfig()
)

brick1_rec1 = HeadsetRecording.from_folder("../example_datasets/brick1/brick1_rec1")
brick1_rec2 = HeadsetRecording.from_folder("../example_datasets/brick1/brick1_rec2")

## Loading Mixed1

In [ ]:
mixed1_scan = CompleteRobotScan.from_folder("../example_datasets/mixed1/mixed1_scan")

mixed1_env = Scanned3dEnvironment.from_gathered_robot_data(
        mixed1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.1, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig()
)

mixed1_rec1 = HeadsetRecording.from_folder("../example_datasets/mixed1/mixed1_rec1")
mixed1_rec2 = HeadsetRecording.from_folder("../example_datasets/mixed1/mixed1_rec2")

## Visualize Loaded Data

In [ ]:
visualize_loaded_data = False
if visualize_loaded_data:
    for headset_data in [round1_rec1, round1_rec2, round1_rec3]:
        visualize_robot_camera_environment_combo(robot_env=round1_env, headset_data=headset_data)

    for headset_data in [brick1_rec1, brick1_rec2]:
        visualize_robot_camera_environment_combo(robot_env=brick1_env, headset_data=headset_data)
        
    for headset_data in [mixed1_rec1, mixed1_rec2]:
        visualize_robot_camera_environment_combo(robot_env=mixed1_env, headset_data=headset_data)

## Prediction & Evaluation

In [ ]:
round_rec2_grader = grader_for_dataset(env=round1_env, headset_recording=round1_rec2)
plot_from_grader(round_rec2_grader, vis=True)
round_rec2_grader.save_results(name="round1_rec2")

In [ ]:
brick_rec1_grader = grader_for_dataset(env=brick1_env, headset_recording=brick1_rec1)
plot_from_grader(brick_rec1_grader, vis=True)
brick_rec1_grader.save_results(name="brick1_rec1")

In [ ]:
brick_rec2_grader = grader_for_dataset(env=brick1_env, headset_recording=brick1_rec2)
plot_from_grader(brick_rec2_grader)
brick_rec2_grader.save_results(name="brick1_rec2")

In [ ]:
mixed_rec1_grader = grader_for_dataset(env=mixed1_env, headset_recording=mixed1_rec1)
plot_from_grader(mixed_rec1_grader, vis=True)
mixed_rec1_grader.save_results(name="mixed1_rec1")

In [ ]:
mixed_rec2_grader = grader_for_dataset(env=mixed1_env, headset_recording=mixed1_rec2)
plot_from_grader(mixed_rec2_grader, vis= True)
mixed_rec2_grader.save_results(name="mixed1_rec2")